In [1]:
import torch
import sys
print("Interpreter:", sys.executable)
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

Interpreter: d:\maga25\VKRTimeSeries\.venv\Scripts\python.exe
CUDA available: True
GPU name: NVIDIA GeForce GTX 1660


In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используемое устройство: {device}")
if device.type == 'cuda':
    print(f"Видеокарта: {torch.cuda.get_device_name(0)}")
    print(f"Всего видеопамяти: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Используемое устройство: cuda
Видеокарта: NVIDIA GeForce GTX 1660
Всего видеопамяти: 6.44 GB


In [3]:
import random

import numpy as np
import pandas as pd
import torch

from transformers import (
    EarlyStoppingCallback,
    PatchTSTConfig,
    PatchTSTForPrediction,
    Trainer,
    TrainingArguments,
)

from tsfm_public.toolkit.dataset import ForecastDFDataset
from tsfm_public.toolkit.time_series_preprocessor import TimeSeriesPreprocessor
from tsfm_public.toolkit.util import select_by_index

d:\maga25\VKRTimeSeries\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# воспроизводимость
SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

------------------------------------------------------------------------


Дообучение на датасете Weather

In [ ]:
import os

dataset = "data/weather.csv"

num_workers = 6 # кол-во потоков для загрузки данных
batch_size = 32 # кол-во одновременно обрабатываемых образцов данных
context_length = 168 # длина окна
forecast_horizon = 96 # длина предсказания
patch_length = 12 # размер патча (число временных шагов по всем признакам)
pin_memory = True
fp16 = True

timestamp_column = "date"
id_columns = []

train_start_index = None
train_end_index = 9 * 30 * 24 # в обучении 9 месяцев данных (9мес * 30дн * 24ч * 6мин)

# индекс начала валидации сдвинут на context_length, чтобы оценить прогноз с начала валидационных данных
valid_start_index = 9 * 30 * 24 - context_length
valid_end_index = 9 * 30 * 24 + 1 * 30 * 24

# индекс начала теста сдвинут на context_length, чтобы оценить прогноз с начала тестовых данных
test_start_index = 9 * 30 * 24 + 1 * 30 * 24 - context_length
test_end_index = 9 * 30 * 24 + 3 * 30 * 24


# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
data = pd.read_csv(
    dataset,
    parse_dates=[timestamp_column], # тип date -> datetime
)
data.set_index(timestamp_column, inplace=True)

# Ресемплинг на 1 час (среднее)
# Используем '1h' вместо '1H' для совместимости
try:
    data = data.resample('1h').mean().dropna()
except ValueError:
    data = data.resample('60T').mean().dropna()  # альтернативный вариант

data.reset_index(inplace=True)

print(len(data))

# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

forecast_columns = [col for col in data.columns if col != 'date']

train_data = select_by_index(
    data,
    id_columns=id_columns,
    start_index=train_start_index,
    end_index=train_end_index,
)
valid_data = select_by_index(
    data,
    id_columns=id_columns,
    start_index=valid_start_index,
    end_index=valid_end_index,
)
test_data = select_by_index(
    data,
    id_columns=id_columns,
    start_index=test_start_index,
    end_index=test_end_index,
)

tsp_weather = TimeSeriesPreprocessor(
    timestamp_column=timestamp_column,
    id_columns=id_columns,
    target_columns=forecast_columns,
    scaling=True,
)
tsp_weather.train(train_data) # вычисление среднего и дисперсии


train_dataset = ForecastDFDataset(
    tsp_weather.preprocess(train_data),
    id_columns=id_columns,
    target_columns=forecast_columns,
    context_length=context_length,
    prediction_length=forecast_horizon,
    stride=1
)
valid_dataset = ForecastDFDataset(
    tsp_weather.preprocess(valid_data),
    id_columns=id_columns,
    target_columns=forecast_columns,
    context_length=context_length,
    prediction_length=forecast_horizon,
    stride=1
)
test_dataset = ForecastDFDataset(
    tsp_weather.preprocess(test_data),
    id_columns=id_columns,
    target_columns=forecast_columns,
    context_length=context_length,
    prediction_length=forecast_horizon,
    stride=1
)


model_path = "./best_patchtst_etth1_model"
old_config = PatchTSTConfig.from_pretrained(model_path)

# описание архитектуры новой модели с учетом загруженной
new_config = PatchTSTConfig(
    do_mask_input=False,
    num_input_channels=len(forecast_columns),
    context_length=context_length,
    patch_length=patch_length,
    prediction_length=forecast_horizon,
    patch_stride=old_config.patch_length, 
    d_model=128,
    num_attention_heads=old_config.num_attention_heads,
    num_hidden_layers=old_config.num_hidden_layers,
    ffn_dim=old_config.ffn_dim,
    dropout=old_config.dropout,
    head_dropout=old_config.head_dropout,
    pooling_type=old_config.pooling_type,
    channel_attention=old_config.channel_attention,
    scaling=old_config.scaling,
    loss=old_config.loss,
    pre_norm=old_config.pre_norm,
    norm_type=old_config.norm_type
)
model = PatchTSTForPrediction.from_pretrained(
    model_path,
    config=new_config,
    ignore_mismatched_sizes=True,
    device_map="cuda" if torch.cuda.is_available() else "cpu"
)


# параметры обучения
train_args = TrainingArguments(
    output_dir="./checkpoint/patchtst/finetune/weather_full/",
    learning_rate=0.0002,
    num_train_epochs=50,
    do_eval=True,
    eval_strategy="epoch",
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    dataloader_num_workers=6,
    logging_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    label_names=["future_values"],
    fp16=fp16,
    dataloader_pin_memory=pin_memory,
    logging_steps=50
)

# ранняя остановка обучения
early_stopping = EarlyStoppingCallback(
    early_stopping_patience=5,
    early_stopping_threshold=0.001,
)

trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    callbacks=[early_stopping],
)


print("\n\nОбучение на Weather")
trainer.train()

# оценка на тесте
test_metrics = trainer.evaluate(test_dataset)
print(f"Ошибка на тесте: {test_metrics['eval_loss']:.4f}")

8784


Loading weights: 100%|██████████| 71/71 [00:00<00:00, 4176.25it/s]
[transformers] PatchTSTForPrediction LOAD REPORT from: ./best_patchtst_etth1_model
Key                                           | Status   |                                                                                           
----------------------------------------------+----------+-------------------------------------------------------------------------------------------
model.encoder.positional_encoder.position_enc | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([42, 128]) vs model:torch.Size([14, 128])  
head.projection.weight                        | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([96, 5376]) vs model:torch.Size([96, 1792])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.




Обучение на Weather


Epoch,Training Loss,Validation Loss
1,0.543338,0.424804
2,0.516046,0.436028
3,0.506824,0.435006
4,0.496044,0.427063
5,0.486116,0.461931
6,0.474132,0.462074


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 96.99it/s]


Training Loss,Validation Loss,Epoch
0.474132,0.262850,6


Ошибка на тесте: 0.2629


In [6]:
model.save_pretrained("./patchtst_etth1_weather_hourly")
tsp_weather.save_pretrained("./patchtst_etth1_weather_hourly_preprocessor")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 86.70it/s]
INFO:p-23148:t-8544:processor.py:save_pretrained:Feature extractor saved in ./patchtst_etth1_weather_hourly_preprocessor\preprocessor_config.json


['./patchtst_etth1_weather_hourly_preprocessor\\preprocessor_config.json']